SK_ID_PREV, SK_ID_CURR, MONTHS_BALANCE: 前者是同戶的歷史貸款（雜湊後的編號），後者是目前樣本中的貸款編號；MONTHS_BALANCE 為相對於申請日的月份（-1 代表最新月份）。

AMT_BALANCE, AMT_CREDIT_LIMIT_ACTUAL: 該月份上一張信用卡的帳面餘額，以及同月份的實際核准額度。

AMT_DRAWINGS_ATM_CURRENT, AMT_DRAWINGS_POS_CURRENT, AMT_DRAWINGS_OTHER_CURRENT, AMT_DRAWINGS_CURRENT: 依通路區分的提款／刷卡金額；CURRENT 是總提領金額，其餘分別對應 ATM、POS 商品消費及其他通路。

AMT_INST_MIN_REGULARITY, AMT_PAYMENT_CURRENT, AMT_PAYMENT_TOTAL_CURRENT: 當月的最低應繳金額、當月實際繳款、以及當月對該筆信用卡所有繳款總額。

AMT_RECEIVABLE_PRINCIPAL, AMT_RECIVABLE, AMT_TOTAL_RECEIVABLE: 當月尚未收回的本金、整體應收款、與總應收款（含利息等）。

CNT_DRAWINGS_ATM_CURRENT, CNT_DRAWINGS_POS_CURRENT, CNT_DRAWINGS_OTHER_CURRENT, CNT_DRAWINGS_CURRENT, CNT_INSTALMENT_MATURE_CUM: 提領／刷卡的次數（含 ATM、POS、其他及總次數）與累計已還清的期數。

NAME_CONTRACT_STATUS, SK_DPD, SK_DPD_DEF: 當月合約狀態（例如 Active、Signed 等），以及逾期天數（SK_DPD_DEF 允許忽略小額欠款）。

In [4]:
import pandas as pd

credit_card_balance = pd.read_csv('home-credit-default-risk/credit_card_balance.csv')

In [5]:
credit_card_balance.columns

Index(['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE',
       'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT',
       'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT',
       'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY',
       'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT',
       'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE',
       'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT',
       'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT',
       'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD',
       'SK_DPD_DEF'],
      dtype='object')

In [6]:
missing_rate_percent = (credit_card_balance.isnull().mean() * 100).sort_values(ascending=False)
print(missing_rate_percent)

AMT_PAYMENT_CURRENT           19.998063
AMT_DRAWINGS_ATM_CURRENT      19.524872
CNT_DRAWINGS_POS_CURRENT      19.524872
AMT_DRAWINGS_OTHER_CURRENT    19.524872
AMT_DRAWINGS_POS_CURRENT      19.524872
CNT_DRAWINGS_OTHER_CURRENT    19.524872
CNT_DRAWINGS_ATM_CURRENT      19.524872
CNT_INSTALMENT_MATURE_CUM      7.948208
AMT_INST_MIN_REGULARITY        7.948208
SK_ID_PREV                     0.000000
AMT_TOTAL_RECEIVABLE           0.000000
SK_DPD                         0.000000
NAME_CONTRACT_STATUS           0.000000
CNT_DRAWINGS_CURRENT           0.000000
AMT_PAYMENT_TOTAL_CURRENT      0.000000
AMT_RECIVABLE                  0.000000
AMT_RECEIVABLE_PRINCIPAL       0.000000
SK_ID_CURR                     0.000000
AMT_DRAWINGS_CURRENT           0.000000
AMT_CREDIT_LIMIT_ACTUAL        0.000000
AMT_BALANCE                    0.000000
MONTHS_BALANCE                 0.000000
SK_DPD_DEF                     0.000000
dtype: float64


In [7]:
status_counts = credit_card_balance["NAME_CONTRACT_STATUS"].value_counts()
credit_card_balance["NAME_CONTRACT_STATUS"] = credit_card_balance["NAME_CONTRACT_STATUS"].apply(
    lambda x: x if status_counts[x] >= 10000 else "Other"
)
print(credit_card_balance["NAME_CONTRACT_STATUS"].value_counts())

NAME_CONTRACT_STATUS
Active       3698436
Completed     128918
Signed         11058
Other           1900
Name: count, dtype: int64


In [8]:
months_with_dpd = (
    credit_card_balance.groupby("SK_ID_CURR")["SK_DPD_DEF"]
    .apply(lambda x: (x > 0).sum())
    .reset_index(name="months_with_dpd")
)

In [9]:
categorical_cols = ["NAME_CONTRACT_STATUS"]
credit_card_balance = pd.get_dummies(credit_card_balance, columns=categorical_cols, drop_first=False)

# Aggregate numeric and one-hot features to the SK_ID_CURR level
agg_dict = {
    "MONTHS_BALANCE": ["mean"],
    "AMT_BALANCE": ["mean"],
    "AMT_CREDIT_LIMIT_ACTUAL": ["mean"],
    "AMT_DRAWINGS_CURRENT": ["sum"],
    "AMT_DRAWINGS_POS_CURRENT": ["sum"],
    "AMT_PAYMENT_TOTAL_CURRENT": ["sum"],
    "AMT_RECEIVABLE_PRINCIPAL": ["mean"],
    "AMT_TOTAL_RECEIVABLE": ["mean"],
    "CNT_DRAWINGS_CURRENT": ["sum"],
    "CNT_DRAWINGS_POS_CURRENT": ["sum"],
    "CNT_INSTALMENT_MATURE_CUM": ["max"],
    "SK_DPD": ["max"],
    "SK_DPD_DEF": ["max"],
}

for col in credit_card_balance.columns:
    if col.startswith("NAME_CONTRACT_STATUS_"):
        agg_dict[col] = ["sum"]

credit_card_balance_agg = credit_card_balance.groupby("SK_ID_CURR").agg(agg_dict)
credit_card_balance_agg.columns = [
    f"{col}_{stat}" for col, stat in credit_card_balance_agg.columns
]
credit_card_balance_agg = credit_card_balance_agg.reset_index()

credit_card_balance_agg = credit_card_balance_agg.merge(
    months_with_dpd,
    on="SK_ID_CURR",
    how="left",
)

credit_card_balance_agg.head()

,SK_ID_CURR,MONTHS_BALANCE_mean,AMT_BALANCE_mean,AMT_CREDIT_LIMIT_ACTUAL_mean,AMT_DRAWINGS_CURRENT_sum,AMT_DRAWINGS_POS_CURRENT_sum,AMT_PAYMENT_TOTAL_CURRENT_sum,AMT_RECEIVABLE_PRINCIPAL_mean,AMT_TOTAL_RECEIVABLE_mean,CNT_DRAWINGS_CURRENT_sum,CNT_DRAWINGS_POS_CURRENT_sum,CNT_INSTALMENT_MATURE_CUM_max,SK_DPD_max,SK_DPD_DEF_max,NAME_CONTRACT_STATUS_Active_sum,NAME_CONTRACT_STATUS_Completed_sum,NAME_CONTRACT_STATUS_Other_sum,NAME_CONTRACT_STATUS_Signed_sum,months_with_dpd
0,100006,-3.5,0.000000,270000.000000,0.0,0.0,0.000,0.000000,0.000000,0,0.0,0.0,0,0,6,0,0,0,0
1,100011,-38.5,54482.111149,164189.189189,180000.0,0.0,334485.000,52402.088919,54433.179122,4,0.0,33.0,0,0,74,0,0,0,0
2,100013,-48.5,18159.919219,131718.750000,571500.0,0.0,654448.545,17255.559844,18101.079844,23,0.0,22.0,1,1,96,0,0,0,1
3,100021,-10.0,0.000000,675000.000000,0.0,0.0,0.000,0.000000,0.000000,0,0.0,0.0,0,0,7,10,0,0,0
4,100023,-7.5,0.000000,135000.000000,0.0,0.0,0.000,0.000000,0.000000,0,0.0,0.0,0,0,8,0,0,0,0


In [10]:
pd.DataFrame.to_csv(credit_card_balance_agg, "transformed_data/_credit_card_balance.csv") 